# Agentic AI: Langchain 

# 0. Preparation de l'environement 

Dans le dossier TP5, l'environnement est défini par `pyproject.toml` et `uv.lock`.
Pour le recréer, exécutez `uv sync`, puis sélectionnez le noyau Python du dossier
`.venv` dans VS Code.

Les dépendances utilisent LangChain, LangGraph et l'intégration officielle
`langchain-google-genai`. La clé doit rester dans le fichier local `.env` sous la
forme `GEMINI_API_KEY=...`. Ce fichier est exclu du dépôt Git ; ne collez jamais
la clé dans le notebook.

Le notebook utilise `gemini-3.5-flash-lite` pour limiter les délais et prendre
en charge les outils, les réponses structurées et le niveau de réflexion requis
par les exercices. Les appels de modèle nécessitent une connexion Internet et
restent soumis aux limites d'utilisation de Google.

In [43]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


def response_text(message):
    """Extract displayable text while ignoring provider metadata blocks."""
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and isinstance(block.get("text"), str):
                parts.append(block["text"])
        return "\n".join(part for part in parts if part)
    return str(content)


In [18]:
# Find the TP5-local .env whether VS Code starts the kernel from TP5 or the repository root.
cwd = Path.cwd()
env_candidates = (cwd / ".env", cwd / "TP5" / ".env", cwd.parent / "TP5" / ".env")
env_path = next((candidate for candidate in env_candidates if candidate.is_file()), cwd / ".env")
load_dotenv(dotenv_path=env_path, override=True)

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(f"GEMINI_API_KEY absente. Créez le fichier .env local : {env_path}")

GEMINI_MODEL_NAME = "gemini-3.5-flash-lite"
gemini_model = ChatGoogleGenerativeAI(model=GEMINI_MODEL_NAME, timeout=20)
llm = gemini_model
print(f"Client Gemini prêt ({GEMINI_MODEL_NAME}).")


Client Gemini prêt (gemini-3.5-flash-lite).


Agent Simple

In [19]:
agent = create_agent(
    model=gemini_model,
    tools=[],
    system_prompt="You are a helpful assistant. Use only information provided in the conversation.",
)


In [20]:
response = agent.invoke({
    "messages": [HumanMessage(content="My first name is Ahmed and my last name is Miloudi.")]
})


In [14]:
print(display(Markdown(response_text(response['messages'][-1]))))

Hello Ahmed Miloudi! How can I help you today?

None


In [21]:
response = agent.invoke({
    "messages": [HumanMessage(content="I am Ahmed. What is my last name?")]
})


In [22]:
print(display(Markdown(response_text(response['messages'][-1]))))

Based on our conversation, I don't know your last name because you only mentioned your first name is Ahmed.

None


NB: vous pouvez specifier model

Configuration et controle de modele

Pour mieux contrôler la configuration du modèle, initialisez directement une instance du modèle à l'aide du package du fournisseur. Dans cet exemple, nous utilisons ChatOpenAl. Consultez la section Modèles de chat pour découvrir les autres classes de modèles de chat disponibles.

In [23]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage


In [24]:
llm = gemini_model


In [25]:
agent=create_agent(
   model=llm,
   tools=[],
   system_prompt= "You are a helpful assistant",
   debug=True
    )
response =agent.invoke(input={"messages" :[HumanMessage("What is the healthiest time to eat dinner? ")]
})

[values] {'messages': [HumanMessage(content='What is the healthiest time to eat dinner? ', additional_kwargs={}, response_metadata={}, id='d8f330b9-3e13-414e-96c0-f07455d2c691')]}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'From a physiological and digestive standpoint, the healthiest time to eat dinner is generally **between 3:00 PM and 7:00 PM**, or at least **2 to 3 hours before you go to bed**. \n\nHowever, "healthy" timing isn\'t just about the clock on the wall—it’s about how your body processes food and aligns with your internal clock (circadian rhythm). \n\nHere is a breakdown of why timing matters, the ideal window, and tips for making it work with a busy schedule.\n\n---\n\n### Why Dinner Timing Matters\nYour body’s metabolism changes throughout the day. In the morning and early afternoon, your body is primed to burn calories, regulate blood sugar, and handle digestion efficiently. In the evening, your metabolism slows down, and your digesti

In [26]:
print(display(Markdown(response_text(response['messages'][-1]))))

From a physiological and digestive standpoint, the healthiest time to eat dinner is generally **between 3:00 PM and 7:00 PM**, or at least **2 to 3 hours before you go to bed**. 

However, "healthy" timing isn't just about the clock on the wall—it’s about how your body processes food and aligns with your internal clock (circadian rhythm). 

Here is a breakdown of why timing matters, the ideal window, and tips for making it work with a busy schedule.

---

### Why Dinner Timing Matters
Your body’s metabolism changes throughout the day. In the morning and early afternoon, your body is primed to burn calories, regulate blood sugar, and handle digestion efficiently. In the evening, your metabolism slows down, and your digestive system prepares for rest.

Eating too close to bedtime can lead to several health issues:
* **Acid Reflux and Indigestion:** Lying down soon after eating makes it easier for stomach acid to flow back up into the esophagus.
* **Poor Sleep Quality:** Digestion requires energy and keeps your body temperature elevated, which can interfere with the deep, restorative stages of sleep.
* **Blood Sugar and Weight Concerns:** Late-night eating is associated with higher blood sugar levels and, over time, a greater likelihood of weight gain because the body is less efficient at processing glucose at night.

---

### The Ideal Dinner Window
* **The Sweet Spot:** **5:00 PM to 7:00 PM** works best for the average person. It gives your body enough time to digest the meal before your evening wind-down without forcing you to eat uncomfortably early.
* **The "3-Hour Rule":** If your schedule requires you to eat later (e.g., at 9:00 PM), try to ensure there is a **2- to 3-hour gap** between your last bite and your head hitting the pillow.

---

### What if You Have to Eat Late?
Sometimes life gets in the way, and dinner happens at 8:30 or 9:00 PM. If you find yourself eating late, you can minimize the negative health impacts by changing **what** you eat, rather than just *when* you eat:

1. **Keep it light:** Choose a smaller portion. Think of it as a bridge to breakfast, not your heaviest meal of the day.
2. **Focus on protein and veggies:** Opt for lean proteins (like chicken, fish, or tofu) and non-starchy vegetables. 
3. **Avoid heavy carbs and fats:** Heavy, greasy, or overly spicy foods take much longer to digest and are the primary culprits behind heartburn and restless sleep.
4. **Skip the alcohol and sugar:** Both can disrupt your sleep architecture, even if they help you initially fall asleep.

---

### The Bottom Line
The "healthiest" time to eat dinner ultimately depends on your personal sleep schedule. If you go to bed at 10:00 PM, aim to finish dinner by 7:00 PM. If you are a shift worker or night owl who sleeps at 2:00 AM, eating dinner at 9:00 or 10:00 PM is completely fine, provided you maintain that 3-hour buffer before sleep.

None


# Implémentation de l'agent via LangChain : Dynamic Model
Les modèles dynamiques sont sélectionnés à l'exécution en fonction de l'état et du contexte actuels. Cela permet une logique de routage avancée et une optimisation des coûts. Pour utiliser un modèle dynamique, créez un middleware à l'aid: du décorateur ` @wrap_model_call ` qui modifie le modèle dans la requête :

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


In [28]:
basic_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    thinking_level="low",
    timeout=20,
)
advanced_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    thinking_level="high",
    timeout=20,
)

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose a Gemini configuration using the conversation environment."""
    env = request.runtime.context.get("env", "test")
    if env == "prod":
        selected_model = advanced_model
        print("Advanced Gemini configuration selected.")
    else:
        selected_model = basic_model
        print("Low-latency Gemini configuration selected.")
    return handler(request.override(model=selected_model))


In [29]:
agent = create_agent(
    model=basic_model, # Default model
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

In [30]:
agent.invoke( input={'messages':[
  HumanMessage('Test 1')]},
  context={'env': 'test'})

[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='f36f0466-1451-44d4-b1c4-81ddd9bbcea5')]}
Low-latency Gemini configuration selected.
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Test 1 received successfully! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TUYF0pIg6F8d+xI2VonmeZ1EZ3YTBctEvnNrvsAvyPCOLudUJFlEAPX6Oh252TXwR+QQS9kkG5/y94zpnXaKbPL1VOVAaZemQomQh7kZReExjqNIK8dEEiQM='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a188b5-fa38-712a-9b0f-6c7f399d0ff9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 13, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})]}}
[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='f36f0466-1451-44d4-b1c4-81ddd9bbcea5'

{'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='f36f0466-1451-44d4-b1c4-81ddd9bbcea5'),
  AIMessage(content=[{'type': 'text', 'text': 'Test 1 received successfully! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TUYF0pIg6F8d+xI2VonmeZ1EZ3YTBctEvnNrvsAvyPCOLudUJFlEAPX6Oh252TXwR+QQS9kkG5/y94zpnXaKbPL1VOVAaZemQomQh7kZReExjqNIK8dEEiQM='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a188b5-fa38-712a-9b0f-6c7f399d0ff9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 13, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})]}

In [31]:
agent.invoke(
    input={"messages": [HumanMessage(content="Test 1")]},
    context={"env": "prod"},
)


[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='e345c667-95a9-4933-aa12-f932d3cf4b55')]}
Advanced Gemini configuration selected.
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Test 1 received! Loud and clear. How can I help you today?', 'extras': {'signature': 'EqEHCp4HAWkUfRNFCHFQJr7DEL+O//R6KWBHEjTInBsalt9v6EttHnB0/9TFAPnTsT1eYHBX67dYmN98Uxe0PWSn7rz/zxkLsoHONH0h9YJliocOz/pVo+Ek5loXcpH29zPTXwYAYJzlwXd54bspg8dcOljpMssqMPWKZonr9eNsSjg9wA3W5TNJDdFJIr463P1N6P+ha+yKXNipD5nIPr317/1ayeG5j4K6Yx+K4r5wQyYjrRj8dFQLKtswRMlrmoB9S5J0jJgl3ZeNda/jzd46jDyQPEKht46MN1OS7uTZi7zHGnQko9XIR7tlfSTs2eU7zLZpdsBXRDmBOIgYP3PIk/JBF38RU3zh++yz6q2k8dpaXKTLm8rVfkwHS9ipWh2CQzt9CuCHXRf2zasTYDpkbJj1Ma0iWdmw6WYrybnCC3zK/sPVIFZWgu4HsVUk04foJ2Xpmvyn1PEwc710rg/ijGcLHHoP/ou9qpsC0Tx+jrwswuAmVAEuK90EWItjfDxwhuPIcqbEodLUlykNt2aNfs/922kPYlLFYhm/Gym31OkpB65Zrtj26UPZH3nngH7Ughapw0OK5vmvK6hZj8twI6ztYX4aIAdwj3S44JpTwLtvPx29p64v7uHstR9WmKYRuqt2jbIb

{'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='e345c667-95a9-4933-aa12-f932d3cf4b55'),
  AIMessage(content=[{'type': 'text', 'text': 'Test 1 received! Loud and clear. How can I help you today?', 'extras': {'signature': 'EqEHCp4HAWkUfRNFCHFQJr7DEL+O//R6KWBHEjTInBsalt9v6EttHnB0/9TFAPnTsT1eYHBX67dYmN98Uxe0PWSn7rz/zxkLsoHONH0h9YJliocOz/pVo+Ek5loXcpH29zPTXwYAYJzlwXd54bspg8dcOljpMssqMPWKZonr9eNsSjg9wA3W5TNJDdFJIr463P1N6P+ha+yKXNipD5nIPr317/1ayeG5j4K6Yx+K4r5wQyYjrRj8dFQLKtswRMlrmoB9S5J0jJgl3ZeNda/jzd46jDyQPEKht46MN1OS7uTZi7zHGnQko9XIR7tlfSTs2eU7zLZpdsBXRDmBOIgYP3PIk/JBF38RU3zh++yz6q2k8dpaXKTLm8rVfkwHS9ipWh2CQzt9CuCHXRf2zasTYDpkbJj1Ma0iWdmw6WYrybnCC3zK/sPVIFZWgu4HsVUk04foJ2Xpmvyn1PEwc710rg/ijGcLHHoP/ou9qpsC0Tx+jrwswuAmVAEuK90EWItjfDxwhuPIcqbEodLUlykNt2aNfs/922kPYlLFYhm/Gym31OkpB65Zrtj26UPZH3nngH7Ughapw0OK5vmvK6hZj8twI6ztYX4aIAdwj3S44JpTwLtvPx29p64v7uHstR9WmKYRuqt2jbIbz9WECwvaEmnI7f2/A6nmGdgfcgNm0spmsGJ2L1rZVZoBBDAssBJprUUTc20XnfFxwF2/WckFUxjj/hfMNJ

# Agent Memory

In [32]:

from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

In [33]:
agent = create_agent(
    model=gemini_model,
    system_prompt="You are a helpful assistant. Remember information within this thread only.",
    checkpointer=InMemorySaver(),
)


In [35]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="My name is Ahmed Miloudi.")]},
    config={"configurable": {"thread_id": "tp5-memory-demo"}},
)
print(response_text(resp["messages"][-1]))


Nice to meet you, Ahmed Miloudi! I will make sure to call you by that name. How can I help you today?


In [36]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="What name did I give you?")]},
    config={"configurable": {"thread_id": "tp5-memory-demo"}},
)
print(response_text(resp["messages"][-1]))


You told me your name is Ahmed Miloudi.


In [49]:
from pathlib import Path
import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver

# Charger le .env du dossier du notebook, même si le noyau a été démarré avant sa création.
env_candidates = (
    Path.cwd() / ".env",
    Path.cwd() / "TP5" / ".env",
    Path.cwd().parent / "TP5" / ".env",
)
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path is not None:
    load_dotenv(dotenv_path=env_path, override=True)

POSTGRES_URI = os.getenv("TP5_DATABASE_URL")
postgres_response = None

if POSTGRES_URI:
    print(f"TP5_DATABASE_URL chargée depuis {env_path}.")
    with PostgresSaver.from_conn_string(POSTGRES_URI) as checkpointer:
        checkpointer.setup()
        postgres_agent = create_agent(
            model=gemini_model,
            tools=[],
            checkpointer=checkpointer,
        )
        postgres_response = postgres_agent.invoke(
            {"messages": [HumanMessage(content="My name is Ahmed Miloudi.")]},
            config={"configurable": {"thread_id": "tp5-postgres-demo"}},
        )
else:
    print("TP5_DATABASE_URL introuvable dans l’environnement ou dans le fichier .env.")

TP5_DATABASE_URL chargée depuis c:\Users\Lenovo\Desktop\Distributed-Agentic-System\TP5\.env.


In [50]:
if postgres_response is None:
    print("Aucun appel PostgreSQL effectué ; la mémoire InMemorySaver reste disponible plus haut.")
else:
    print(response_text(postgres_response["messages"][-1]))


Nice to meet you, Ahmed Miloudi! Thanks for letting me know. How can I help you today?


# Agent tools

### Le rôle des outils pour les agents

Les outils donnent aux agents la capacité d'agir. Les agents vont au-delà du simple rattachement d'outils au modèle en facilitant :

* **Plusieurs appels d'outils en séquence :** Déclenchés par un seul prompt.
* **Des appels d'outils en parallèle :** Lorsque cela est pertinent.
* **La sélection dynamique des outils :** En fonction des résultats précédents.
* **La logique de nouvelle tentative (*retry*) :** Ainsi que la gestion des erreurs.
* **La persistance de l'état :** Conservation des données entre les appels d'outils.


In [51]:
from langchain.tools import tool
from langchain.agents import create_agent

In [52]:
@tool
def search(query: str) -> str:
    """Return deterministic sample news for the tool-calling demonstration."""
    print(f"Search tool called for: {query}")
    return "Sample result: the demonstration search tool returned local mock data."


@tool
def get_weather(location: str) -> str:
    """Return fictional weather data for a tool-calling demonstration."""
    print(f"Weather tool called for: {location}")
    return f"Demo weather in {location}: sunny, 25 C (fictional sample)."


@tool
def get_employee_info(name: str) -> dict:
    """Return a synthetic employee record; no real personnel data is used."""
    print(f"Employee lookup tool called for: {name}")
    return {"name": name, "salary": 12000, "job": "Developer"}


In [53]:
agent=create_agent(
    model=llm,
    tools=[search, get_weather, get_employee_info]
)

In [54]:
response=agent.invoke(input={'messages':[HumanMessage('What is the weather in BeniMellal' )]})
print(display(Markdown(response_text(response['messages'][-1])) ))

Weather tool called for: BeniMellal


The weather in BeniMellal is currently sunny and 25°C (this is fictional sample data for demonstration purposes).

None


In [56]:
response = agent.invoke({
    "messages": [HumanMessage(content="What is the job of fictional employee Ahmed Miloudi?")]
})
display(Markdown(response_text(response["messages"][-1])))


Employee lookup tool called for: Ahmed Miloudi


The job of Ahmed Miloudi is **Developer**.

In [57]:
response=agent.invoke(input={'messages':[HumanMessage('What are news' )]})
print(display(Markdown(response_text(response['messages'][-1])) ))

Search tool called for: latest news sample


"News" refers to newly received or noteworthy information, typically about recent events happening locally, nationally, or globally. It covers a wide range of topics, including politics, business, science, technology, entertainment, sports, and human interest stories.

News is typically distributed through various media channels, such as:
* **Print:** Newspapers and magazines.
* **Broadcast:** Television and radio news programs.
* **Digital/Online:** News websites, social media, apps, and podcasts.

Is there a specific topic or current event you are looking for information about?

None


In [58]:
response=agent.invoke(input={'messages':[HumanMessage('What are news' )]})
print(display(Markdown(response_text(response['messages'][-1])) ))

Search tool called for: latest news headlines


**News** refers to newly received or noteworthy information, typically about recent events happening locally, nationally, or globally. It is communicated through various media formats—such as television, radio, print (newspapers and magazines), and digital platforms (websites and social media).

News generally covers a wide range of topics, including:
* **Politics:** Government actions, elections, policies, and international relations.
* **Business & Economy:** Markets, inflation, employment trends, and corporate updates.
* **Science & Technology:** New inventions, space exploration, medical breakthroughs, and digital innovations.
* **World Events:** Major conflicts, humanitarian crises, climate events, and diplomatic agreements.
* **Culture & Entertainment:** Arts, movies, music, lifestyle trends, and human-interest stories.
* **Sports:** Game results, tournaments, athlete profiles, and league updates.

If you are looking for specific information or headlines about a particular topic today, feel free to let me know!

None


In [59]:
from ddgs import DDGS

@tool
def web_search(query: str, num_results: int = 5) -> str:
    """Search public web results with DuckDuckGo; return plain text."""
    try:
        results = DDGS().text(query=query, max_results=num_results)
        if not results:
            return f"No results found for: {query}"
        lines = [f"Web results for: {query}"]
        for index, item in enumerate(results, start=1):
            title = item.get("title", "Untitled")
            body = item.get("body", "No description available")
            url = item.get("href", "")
            lines.append(f"{index}. {title} — {body} ({url})")
        return "\n".join(lines)
    except Exception as exc:
        return f"Web search unavailable ({type(exc).__name__}): {exc}"


In [60]:
llm = gemini_model


In [61]:
agent=create_agent(model=llm, tools=[web_search, get_employee_info, get_weather], debug=True)

resp=agent.invoke(input={'messages': [HumanMessage('Search for Agentic RAG tutorials')]})
print(display(Markdown(response_text(resp['messages'][-1]))))

[values] {'messages': [HumanMessage(content='Search for Agentic RAG tutorials', additional_kwargs={}, response_metadata={}, id='62b9f471-c7df-4bbc-9d88-02e1cb8c0c2b')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "Agentic RAG tutorials"}'}, '__gemini_function_call_thought_signatures__': {'call_13483': 'EmAKXgFpFH0Tk3VgOo9/bwKFAAisZwWegsFV1+foa+8Be20Ph/xqPKFqGB7gxsC/eUEAECxLrSWSfSRX4etE8L7XKKS9UWLjzoNYZG2wmYqeIZy9zewgib5m4wiTD8ru57M='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a188e7-ea9d-7205-8a20-ba81abdfd342-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Agentic RAG tutorials'}, 'id': 'call_13483', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 168, 'output_tokens': 20, 'total_tokens': 188, 'input_token_details': {'cache_read': 0}})]}}
[

Here are some popular and highly recommended tutorials for learning and building **Agentic RAG** (Retrieval-Augmented Generation) systems:

1. **[DataCamp: Agentic RAG: Step-by-Step Tutorial With Demo Project](https://www.datacamp.com/tutorial/agentic-rag-tutorial)**  
   * **What you'll learn:** How to build an Agentic RAG pipeline from scratch, integrating local data sources and web scraping to generate context-aware responses.

2. **[LangChain / LangGraph Docs: Build a custom RAG agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/agentic-rag)**  
   * **What you'll learn:** How to fetch and preprocess documents, index them for semantic search, create a retriever tool, and build an agent that decides dynamically when to use that retriever.

3. **[Google Colab: Agentic RAG Notebook by Aymeric Roucher](https://colab.research.google.com/github/cemgundogan/Hugging_Face_misc_tutorials/blob/main/Tutorial_Agentic_RAG.ipynb/)**  
   * **What you'll learn:** Advanced techniques to turbocharge your RAG with query reformulation and self-query mechanisms (recommended if you already understand basic RAG).

4. **[GitHub: Agentic RAG for Dummies](https://github.com/GiovanniPasq/agentic-rag-for-dummies)**  
   * **What you'll learn:** A minimalist approach to building modular, agent-driven RAG systems using LangGraph with clean, easy-to-understand code.

None


In [62]:
from langchain.agents import create_agent
from langchain.tools import tool
from datetime import datetime

@tool
def search(query: str) -> str:
    """Use the notebook's DuckDuckGo search tool; no Tavily key is needed."""
    return web_search.invoke({"query": query, "num_results": 5})


In [ ]:
# The Gemini client was initialized after loading .env in the setup section.


In [ ]:
model = gemini_model

agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a helpful assistant. Use the search tool when current public information is needed. "
        f"Today's date is {datetime.now().strftime('%Y-%m-%d')}."
    ),
)


In [65]:
resp=agent.invoke(input={'messages': [HumanMessage("Quel temps fait-il aujourd'hui à BeniMellal")]})
print(display(Markdown( response_text(resp['messages'][-1])) ))

Je n'ai pas accès en temps réel aux données météorologiques exactes de la minute, mais d'après les prévisions actuelles pour la région de Béni Mellal, le temps est généralement nuageux à très chargé, avec des températures oscillant globalement entre 18 °C et 26 °C au cours de la journée. 

Pour obtenir les conditions précises de l'heure actuelle (température exacte, vent, humidité), je vous invite à consulter une application météo ou un site spécialisé comme La Chaîne Météo ou Météo Maroc.

None


# PythonREPL Tool

In [66]:
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()
python_repl.run('print(f"La somme de 5 et 6 est {5 + 6}")')


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20836\4072947773.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL
Python REPL can execute arbitrary code. Use with caution.


'La somme de 5 et 6 est 11\n'

In [67]:
import ast
import contextlib
import io
import math

from langchain_core.tools import tool

_SAFE_CALLS = {
    "print": print,
    "sqrt": math.sqrt,
    "abs": abs,
    "min": min,
    "max": max,
    "sum": sum,
    "sorted": sorted,
    "len": len,
    "list": list,
    "tuple": tuple,
    "zip": zip,
    "round": round,
}
_SAFE_NODES = (
    ast.Module, ast.Expr, ast.Assign, ast.Name, ast.Constant,
    ast.List, ast.Tuple, ast.BinOp, ast.UnaryOp, ast.Call,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow,
    ast.UAdd, ast.USub, ast.Load, ast.Store, ast.JoinedStr,
    ast.FormattedValue, ast.ListComp, ast.comprehension, ast.keyword,
)

def safe_python_repl(code: str) -> str:
    """Run a small arithmetic-only Python subset; imports and I/O are rejected."""
    if len(code) > 3000:
        raise ValueError("Code is longer than the safe demo limit.")
    tree = ast.parse(code, mode="exec")
    for node in ast.walk(tree):
        if not isinstance(node, _SAFE_NODES):
            raise ValueError(f"Unsupported Python construct: {type(node).__name__}")
        if isinstance(node, ast.Name) and node.id.startswith("__"):
            raise ValueError("Private and dunder names are not allowed.")
        if isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Name) or node.func.id not in _SAFE_CALLS:
                raise ValueError("Only the documented safe arithmetic functions may be called.")
        if isinstance(node, ast.Assign):
            if any(not isinstance(target, ast.Name) or target.id.startswith("__") for target in node.targets):
                raise ValueError("Assignments are restricted to ordinary variable names.")
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
            if not isinstance(node.right, ast.Constant) or not isinstance(node.right.value, (int, float)) or abs(node.right.value) > 12:
                raise ValueError("Exponent is outside the safe demo limit.")

    output = io.StringIO()
    namespace = {"__builtins__": {}, **_SAFE_CALLS}
    with contextlib.redirect_stdout(output):
        exec(compile(tree, "<safe-python-repl>", "exec"), namespace, namespace)
    return output.getvalue().strip() or "Safe Python demo completed without printed output."


@tool
def repl_tool(code: str) -> str:
    """Run arithmetic and list examples in a restricted Python subset (no imports, files, or network)."""
    return safe_python_repl(code)


In [68]:
repl_tool.invoke("""
a=5
b=9
print(f"la somme de {a} et {b} est {a+b}")
""")

'la somme de 5 et 9 est 14'

In [69]:
llm = gemini_model


In [70]:
agent = create_agent(
    model=llm,
    tools=[repl_tool],
    debug=True,
    system_prompt=(
        "Use repl_tool only for arithmetic and list calculations. "
        "Do not attempt imports, filesystem, network, or operating-system access. "
        "Print the result."
    ),
)


In [71]:
resp= agent.invoke(input={'messages':[HumanMessage(
"""
a= 5
b=9
print(f"la somme de {a} et {b} est {a+b}")
"""
)]})
print(display(Markdown(response_text(resp['messages'][-1]))))

[values] {'messages': [HumanMessage(content='\na= 5\nb=9\nprint(f"la somme de {a} et {b} est {a+b}")\n', additional_kwargs={}, response_metadata={}, id='50c83c16-3688-4068-ab39-1af1420b881e')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'repl_tool', 'arguments': '{"code": "a = 5\\nb = 9\\nprint(f\\"la somme de {a} et {b} est {a+b}\\")\\n"}'}, '__gemini_function_call_thought_signatures__': {'call_52581': 'EmAKXgFpFH0TWfubYFFf4TxiYiQKJZZWcrVsDt0Z+/bVsXeIbEkeBK8QQmSOx6Na4iha7w6SPBvF4LTLrjEv6M3Duw//TpDo+N9R9vcmJX2fJhXIbxGoleuCfbh59V41Py8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a188ea-b136-70a7-b822-5702bb8f5910-0', tool_calls=[{'name': 'repl_tool', 'args': {'code': 'a = 5\nb = 9\nprint(f"la somme de {a} et {b} est {a+b}")\n'}, 'id': 'call_52581', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

Le code affichera :

```text
la somme de 5 et 9 est 14
```

None


In [73]:
resp = agent.invoke({
    "messages": [HumanMessage(content=(
        "Utilise repl_tool et exécute ce calcul avec les fonctions autorisées "
        "sorted, zip et print. N'utilise aucune méthode comme .sort(). "
        "list1 = sorted([5, 3, 8])\n"
        "list2 = sorted([1, 9, 3])\n"
        "result = [a + b for a, b in zip(list1, list2)]\n"
        "print(result)"
    ))]
})
display(Markdown(response_text(resp["messages"][-1])))

[values] {'messages': [HumanMessage(content="Utilise repl_tool et exécute ce calcul avec les fonctions autorisées sorted, zip et print. N'utilise aucune méthode comme .sort(). list1 = sorted([5, 3, 8])\nlist2 = sorted([1, 9, 3])\nresult = [a + b for a, b in zip(list1, list2)]\nprint(result)", additional_kwargs={}, response_metadata={}, id='ac2d85b3-02ec-4936-aeec-344578f520d0')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'repl_tool', 'arguments': '{"code": "list1 = sorted([5, 3, 8])\\nlist2 = sorted([1, 9, 3])\\nresult = [a + b for a, b in zip(list1, list2)]\\nprint(result)"}'}, '__gemini_function_call_thought_signatures__': {'call_101925': 'EmAKXgFpFH0TlgpUndcy61AtQtfe1WVfsfdSZpeH+lJ3MfuSE6h+dCfyKhY/bDkdUKuRQVC7MEfSdO+X0BKwURzWrEnele0rXJzWiU2wQIaYfEUWJmHNlgNN9pwqxCsAPs8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01

Le résultat du calcul est :
[4, 8, 17]

In [74]:
from langchain.messages import SystemMessage, HumanMessage


In [75]:
agent4 = create_agent(
    model=llm,
    tools=[repl_tool],
    system_prompt=(
        "Use repl_tool to calculate the requested polynomial roots with arithmetic and sqrt. "
        "Do not import packages or access files, the network, or the operating system. Print the result."
    ),
    debug=True,
)


In [77]:
resp = agent4.invoke({
    "messages": [HumanMessage(content=(
        "Use repl_tool. Do not use import statements or modules. "
        "Call the allowed function sqrt directly, not math.sqrt. "
        "Use only assignments, arithmetic, sqrt, round, and print. "
        "Execute this calculation:\n"
        "a = 6\n"
        "b = 11\n"
        "c = -6\n"
        "delta = b**2 - 4*a*c\n"
        "x1 = (-b + sqrt(delta)) / (2*a)\n"
        "x2 = (-b - sqrt(delta)) / (2*a)\n"
        "Print the quadratic formula, delta, and both roots rounded to 4 decimals."
    ))]
})
display(Markdown(response_text(resp["messages"][-1])))

[values] {'messages': [HumanMessage(content='Use repl_tool. Do not use import statements or modules. Call the allowed function sqrt directly, not math.sqrt. Use only assignments, arithmetic, sqrt, round, and print. Execute this calculation:\na = 6\nb = 11\nc = -6\ndelta = b**2 - 4*a*c\nx1 = (-b + sqrt(delta)) / (2*a)\nx2 = (-b - sqrt(delta)) / (2*a)\nPrint the quadratic formula, delta, and both roots rounded to 4 decimals.', additional_kwargs={}, response_metadata={}, id='8a42f089-2887-422e-a53a-8d18bd89917d')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'repl_tool', 'arguments': '{"code": "a = 6\\nb = 11\\nc = -6\\ndelta = b**2 - 4*a*c\\nx1 = (-b + sqrt(delta)) / (2*a)\\nx2 = (-b - sqrt(delta)) / (2*a)\\n\\nprint(\\"Quadratic formula: a =\\", a, \\", b =\\", b, \\", c =\\", c)\\nprint(\\"delta =\\", delta)\\nprint(\\"x1 =\\", round(x1, 4))\\nprint(\\"x2 =\\", round(x2, 4))\\n"}'}, '__gemini_function_call_thought_signatures__': 

Quadratic formula: a = 6 , b = 11 , c = -6
delta = 265
x1 = 0.4399
x2 = -2.2732

In [78]:
print(response_text(resp["messages"][-1]))


Quadratic formula: a = 6 , b = 11 , c = -6
delta = 265
x1 = 0.4399
x2 = -2.2732


# Tool handlyng errr 

In [80]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
@wrap_tool_call
def handle_tool_errors(request, handler):
   """ Handl tool execution errors with custom messages."""
   try:
      return handler(request)
   except Exception as e:
     print ('Errorrrr')
     # Return a custom error message to the model
     return ToolMessage(
        content=f"Tool error: Please check your input and try again. ({str(e)})",
        tool_call_id=request.tool_call["id"]
    )
agent = create_agent(
    model=gemini_model,
    tools=[search, get_weather],
    middleware=[handle_tool_errors], debug=True
)

# controle de System Prompt

In [81]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate a system prompt based on the caller's role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."
    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    if user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."
    return base_prompt


In [82]:
agent =create_agent(
    model=gemini_model, tools=[],
    middleware= [user_role_prompt],
    context_schema=Context,
    debug=True
)

In [83]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain AI agent"}]},
    context={"user_role": "expert"})
print(display(Markdown(response_text(result['messages'][-1])) ))

[values] {'messages': [HumanMessage(content='Explain AI agent', additional_kwargs={}, response_metadata={}, id='d6a23a86-7c66-4231-9007-d7b9897e3f88')]}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'An **AI agent** is an autonomous software entity that perceives its environment through sensors, makes decisions using artificial intelligence techniques, and takes actions through actuators to achieve a specific goal. \n\nUnlike traditional software that strictly follows pre-written, deterministic IF-THEN rules, or passive machine learning models that simply classify data or generate text when prompted, an AI agent is designed to **reason, plan, and execute multi-step workflows** with minimal human intervention.\n\nHere is a detailed breakdown of how AI agents work, their core architecture, and their significance in the current technological landscape.\n\n---\n\n### 1. The Core Architecture of an AI Agent\n\nAt a conceptual level, an AI agent operates on a 

An **AI agent** is an autonomous software entity that perceives its environment through sensors, makes decisions using artificial intelligence techniques, and takes actions through actuators to achieve a specific goal. 

Unlike traditional software that strictly follows pre-written, deterministic IF-THEN rules, or passive machine learning models that simply classify data or generate text when prompted, an AI agent is designed to **reason, plan, and execute multi-step workflows** with minimal human intervention.

Here is a detailed breakdown of how AI agents work, their core architecture, and their significance in the current technological landscape.

---

### 1. The Core Architecture of an AI Agent

At a conceptual level, an AI agent operates on a continuous loop: **Perceive $\rightarrow$ Think $\rightarrow$ Act**. 

```
[ Environment ] ---> (Sensors) ---> [ AI Agent: Perceive ]
                                           |
                                     [ Think / Reason ]
                                           |
                                    [ Plan / Memory ]
                                           |
[ Environment ] <--- (Actuators) <--- [ Act / Execute ]
```

1. **Perception (Sensors):** The agent takes in data from its environment. In a video game, this might be the pixel data on the screen. For a software engineering agent, it might be a GitHub issue description, a codebase, and error logs. For a Large Language Model (LLM) agent, the "perception" is typically the user's prompt and any retrieved context.
2. **Reasoning & Planning (The "Brain"):** This is where AI (traditionally symbolic AI, reinforcement learning, or modern LLMs) comes in. The agent processes the input, evaluates its ultimate goal, consults its memory, and determines the best sequence of steps to take.
3. **Action (Actuators):** The agent executes its decision. This could be moving a robotic arm, writing code to a file, executing an API call, or generating a text response to a user.

---

### 2. Key Components of Modern LLM-Based Agents

With the advent of powerful foundation models (like GPT-4, Claude 3.5, or open-source equivalents), AI agents have evolved beyond narrow domain scripts into generalized problem solvers. A modern LLM-based agent typically consists of four key modules:

* **The LLM as the Core Engine:** Acts as the "brain," handling natural language understanding, reasoning, task decomposition, and decision-making.
* **Planning:** 
  * *Subgoal Decomposition:* Breaking a large, complex goal into smaller, manageable subtasks (e.g., "Write an app" becomes "1. Design schema, 2. Write backend, 3. Write frontend, 4. Test").
  * *Reflection and Self-Correction:* The agent reviews its own work, catches errors (e.g., a syntax error in generated code), and adjusts its plan accordingly.
* **Memory:**
  * *Short-term/Working Memory:* The immediate context window of the conversation or current task steps.
  * *Long-term Memory:* External vector databases or knowledge graphs that allow the agent to retain information over long periods, recall past interactions, and learn from experience.
* **Tools (Action Space):** Agents are given access to external tools to extend their capabilities beyond text generation. These include:
  * Web browsers/search engines.
  * Python REPLs (code execution environments).
  * APIs (database queries, email senders, CRM software).

---

### 3. Types of AI Agents

In computer science (specifically following Russell and Norvig’s *Artificial Intelligence: A Modern Approach*), agents are generally categorized by their complexity:

1. **Simple Reflex Agents:** Act purely on condition-action rules (e.g., a thermostat that turns on the heater if the temperature drops below 68°F). They cannot handle environments with unobserved states.
2. **Model-Based Reflex Agents:** Maintain an internal state to track aspects of the world that cannot be seen right now (e.g., a self-driving car tracking cars in its blind spot).
3. **Goal-Based Agents:** Have explicit goals and choose actions that explicitly lead toward those goals. They involve search and planning algorithms.
4. **Utility-Based Agents:** Not only want to achieve a goal, but want to do so in the *best possible way* (optimizing a "utility function," such as maximizing speed while minimizing fuel consumption).
5. **Learning Agents:** Can improve their performance over time through experience and adaptation (heavily utilizing Reinforcement Learning).

---

### 4. Examples of AI Agents

* **Autonomous Vehicles (e.g., Waymo):** Perceive the road via LiDAR and cameras, plan trajectories, and actuate steering and braking.
* **Coding Agents (e.g., Devin, GitHub Copilot Workspace):** Given a prompt like "Fix this security vulnerability," they explore the codebase, write patches, run unit tests, and submit pull requests autonomously.
* **Customer Support Agents:** Handle complex multi-turn issues, access backend databases to check order statuses, process refunds, and escalate to humans only when necessary.
* **Personal Assistants (Future state):** Book flights, coordinate schedules across multiple calendars, and buy groceries based on dietary preferences and inventory tracking.

---

### 5. Why AI Agents Matter (The Current Shift)

For the past few years, AI has largely been **interactive and transactional**: you ask a question, and it gives an answer. 

AI agents represent a shift toward **autonomous delegation**. Instead of a human executing a 20-step workflow where the AI assists in each step, a human gives an agent a high-level goal, and the agent executes the 20 steps end-to-end. This promises massive productivity gains, though it introduces new engineering challenges regarding reliability, security, infinite loops, and alignment (ensuring the agent doesn't achieve its goal in unintended, harmful ways).

None


In [84]:
result = agent.invoke(
{"messages": [{"role": "user", "content": "Explain AI Agent"}]},
context={"user_role": "beginner"})
print(display(Markdown(response_text(result['messages'][-1])) ))

[values] {'messages': [HumanMessage(content='Explain AI Agent', additional_kwargs={}, response_metadata={}, id='6b09fe7b-8daa-4789-9c81-47008f79529c')]}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Think of an **AI Agent** as a computer program that has a specific goal, can look at what’s happening around it, and can take actions on its own to reach that goal. \n\nIf regular AI is like a **brain in a jar** that just answers questions when you type to it, an AI Agent is like a **robot with hands and eyes** that can actually go out and *do* things.\n\nHere are the three main parts of an AI Agent:\n\n1. **The Senses (Perception):** It takes in information. This could be text you type, data from the internet, or even a camera feed.\n2. **The Brain (Reasoning):** It thinks about the information, figures out what it means, and decides on the best next step to reach its goal.\n3. **The Hands (Action):** It does something. It might write an email, book a fligh

Think of an **AI Agent** as a computer program that has a specific goal, can look at what’s happening around it, and can take actions on its own to reach that goal. 

If regular AI is like a **brain in a jar** that just answers questions when you type to it, an AI Agent is like a **robot with hands and eyes** that can actually go out and *do* things.

Here are the three main parts of an AI Agent:

1. **The Senses (Perception):** It takes in information. This could be text you type, data from the internet, or even a camera feed.
2. **The Brain (Reasoning):** It thinks about the information, figures out what it means, and decides on the best next step to reach its goal.
3. **The Hands (Action):** It does something. It might write an email, book a flight, move a physical robot arm, or delete spam files.

### A Simple Analogy: A Roomba vs. A Smart Assistant

* **A standard AI chatbot** is like a smart person sitting at a desk. You ask them for a recipe, and they write one down for you. But they can't go into the kitchen and cook it.
* **An AI Agent** is more like a personal assistant. You say, "Order groceries for taco night," and the agent goes to the website, picks the ingredients, pays for them, and has them delivered to your door without needing you to do the steps in between.

### Why are people excited about them?
Right now, AI usually requires a human to guide it step-by-step. AI Agents are the next step because they can work **autonomously** (on their own) for long stretches of time to finish complicated, multi-step tasks.

None


# Structured output

In [87]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=gemini_model,
    tools=[],
    response_format=ToolStrategy(ContactInfo),
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract the contact info from this fictional example: Ahmed Miloudi, ahmed@example.com, +212-555-0147",
    }]
})
contact = result["structured_response"]


In [88]:
print(contact.name)
print(contact.email)
print(contact.phone)

Ahmed Miloudi
ahmed@example.com
+212-555-0147


# Custom Middleware

In [89]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.messages import HumanMessage
import time

class HooksDemo(AgentMiddleware):
    def __init__(self) :
     super().__init__()
     self.starttime = 0.0
    def before_agent(self, state: AgentState, runtime):
        self.starttime = time.time()
        print('Befor_agent triggered')
    def before_model(self, state : AgentState, runtime):
        print ('Beforre_model triggered')
    def after_model (self, state: AgentState, runtime):
        print('After Model triggered')
    def after_agent(self, state : AgentState, runtime):
        print( 'After Agent triggered')
        print("duration =", time. time()-self.starttime)

In [90]:
agent = create_agent(model=gemini_model, middleware=[HooksDemo()])
res = agent.invoke({"messages": [HumanMessage(content="Name three well-known cities in Morocco.")]})


Befor_agent triggered
Beforre_model triggered
After Model triggered
After Agent triggered
duration = 1.0896563529968262


# RAG As Agent Tool

In [91]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20836\861052689.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [92]:
texts = [
    "A is a fictional computer science lecturer.",
    "Ahmed works at a fictional university in Morocco.",
    "Ahmed completed a fictional doctorate in computer science in 2025.",
    "Ahmed enjoys music, culture, and philosophy.",
    "Ahmed grew up in the fictional town of Tazrout.",
    "Ahmed studied computer science at two fictional Moroccan institutions.",
]


In [93]:
vector_store = FAISS.from_texts(texts=texts, embedding=embedding_model)
results = vector_store.similarity_search("Name and affiliation", k=2)


In [94]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
@wrap_tool_call
def handle_tool_errors(request, handler):
   """ Handl tool execution errors with custom messages."""
   try:
      return handler(request)
   except Exception as e:
     print ('Errorrrr')
     # Return a custom error message to the model
     return ToolMessage(
        content=f"Tool error: Please check your input and try again. ({str(e)})",
        tool_call_id=request.tool_call["id"]
    )
agent = create_agent(
    model=gemini_model,
    tools=[search, get_weather],
    middleware=[handle_tool_errors], debug=True
)

In [95]:
print(results[0].page_content)
print(results[1].page_content)


A is a fictional computer science lecturer.
Ahmed works at a fictional university in Morocco.


In [97]:
from langchain_core.tools import create_retriever_tool

retrieval = vector_store.as_retriever(search_kwargs={"k": 5})

retrieval_tool = create_retriever_tool(
    retriever=retrieval,
    name="kb_search",
    description="Search information about me"
)

In [98]:
search_agent = create_agent(
    model=gemini_model,
    system_prompt="Answer only from the fictional profile passages returned by the retrieval tool.",
    tools=[retrieval_tool],
)


In [99]:
result = search_agent.invoke({
    "messages": [HumanMessage(content="What is the fictional profile's name and affiliation?")]
})
display(Markdown(response_text(result["messages"][-1])))


Based on the fictional profile, the name is Ahmed, and he is affiliated with a fictional university in Morocco as a computer science lecturer.

# Guardrails
Les garde-fous vous aident à concevoir des applications d'IA sûres et conformes en validant et en filtrant le contenu à des étapes clés de l'exécution de votre agent. Ils peuvent détecter des informations sensibles, appliquer des politiques de contenu, valider les sorties et prévenir les comportements dangereux avant qu'ils ne causent des problèmes.

**Les cas d'usage courants incluent :**

* Empêcher la fuite de données personnelles identifiables (PII)
* Détecter et bloquer les attaques par injection de prompt
* Bloquer les contenus inappropriés ou nuisibles
* Appliquer les règles métier et les exigences de conformité
* Valider la qualité et l'exactitude des résultats

Vous pouvez mettre en œuvre des garde-fous à l'aide d'un middleware pour intercepter l'exécution à des points stratégiques — avant le démarrage de l'agent, après son exécution, ou autour des appels au modèle et aux outils.

Les garde-fous peuvent être mis en œuvre selon deux approches complémentaires :

* **Garde-fous déterministes :** Ils utilisent une logique fondée sur des règles, comme des expressions régulières (regex), la correspondance de mots-clés ou des vérifications explicites. Ils sont rapides, prévisibles et économiques, mais peuvent manquer certaines violations plus subtiles.
* **Garde-fous fondés sur des modèles :** Ils utilisent des LLM ou des classificateurs pour évaluer le contenu avec une compréhension sémantique. Ils permettent de détecter des problèmes subtils que les règles seules ne repèrent pas, mais sont plus lents et plus coûteux.

LangChain fournit à la fois des garde-fous intégrés (par exemple, la détection de PII et l'intervention humaine dans la boucle) ainsi qu'un système de middleware flexible pour créer des garde-fous personnalisés selon l'une ou l'autre approche.

# Garde-fous intégrés

## Détection de PII

LangChain fournit un middleware intégré pour détecter et traiter les informations personnelles identifiables (PII) dans les conversations. Ce middleware peut détecter les types de PII courants, comme les adresses e-mail, les numéros de carte bancaire, les adresses IP, et bien d'autres.

L'image ci-dessous illustre le flux d'exécution et les différents points d'interception (hooks) où ces garde-fous peuvent être appliqués :

![Schéma des points d'interception des garde-fous](image_1.jpeg)

Le middleware de détection des PII est utile dans des cas tels que les applications de santé et de finance soumises à des exigences de conformité, les agents de service client qui doivent assainir les journaux, ainsi que, plus généralement, toute application manipulant des données utilisateur sensibles.

Le middleware PII prend en charge plusieurs stratégies pour traiter les PII détectées :

| Stratégie | Description | Exemple |
| :--- | :--- | :--- |
| `redact` | Remplace par `[REDACTED_{PII_TYPE}]` | `[REDACTED_EMAIL]` |
| `mask` | Masque partiellement l'information (par ex. les 4 derniers chiffres) | `****-****-****-1234` |
| `hash` | Remplace par un hachage déterministe | `a8f5f167...` |
| `block` | Déclenche une exception lorsqu'une PII est détectée | Erreur levée |

### Types de PII intégrés

* `email` — Adresses e-mail
* `credit_card` — Numéros de carte bancaire (validés avec l'algorithme de Luhn)
* `ip` — Adresses IP
* `mac_address` — Adresses MAC
* `url` — URL

# Options de configuration

| Paramètre | Description | Valeur par défaut |
| :--- | :--- | :--- |
| `pii_type` | Type de PII à détecter (intégré ou personnalisé) | Obligatoire |
| `strategy` | Façon de traiter les PII détectées (`"block"`, `"redact"`, `"mask"`, `"hash"`) | `"redact"` |
| `detector` | Fonction de détection personnalisée ou motif regex | `None` (utilise le détecteur intégré) |
| `apply_to_input` | Vérifie les messages utilisateur avant l'appel au modèle | `True` |
| `apply_to_output` | Vérifie les messages de l'IA après l'appel au modèle | `False` |
| `apply_to_tool_results` | Vérifie les messages de résultat des outils après leur exécution | `False` |

In [100]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

@tool
def card_tool(credit_card: str) -> str:
    """Simulate payment handling without storing or echoing the full card."""
    digits = "".join(character for character in credit_card if character.isdigit())
    suffix = digits[-4:] if digits else "----"
    return f"Simulation only: card ending in {suffix}; no payment was made."

@tool
def send_email(email_address: str) -> str:
    """Simulate an email action locally; no email is sent."""
    return f"Simulation only: an email action was prepared for {email_address}; nothing was sent."

@tool
def process_api_key(api_key: str) -> str:
    """Simulate secret handling without logging or returning the secret."""
    return "Simulation only: a secret value was received and redacted."

agent = create_agent(
    model=gemini_model,
    tools=[card_tool, send_email],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(
            "api_key",
            detector=r"(?:sk-[A-Za-z0-9_-]{20,}|AIza[0-9A-Za-z_-]{20,}|AQ\.[A-Za-z0-9_-]{20,})",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)


In [101]:
# Synthetic test data: the PII middleware should redact the email and mask the test card.
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is ahmed@example.com and my test card is 4242 4242 4242 4242.",
    }]
})


In [102]:
print(response_text(result['messages'][-1]))

Thanks for sharing that! How can I help you today with your test card or email?


# Intervention humaine dans la boucle (Human-in-the-loop)

LangChain fournit un middleware intégré permettant d'exiger une approbation humaine avant l'exécution d'opérations sensibles. Il s'agit de l'un des garde-fous les plus efficaces pour les décisions à forts enjeux.

Le middleware d'intervention humaine dans la boucle est utile dans des cas tels que :
* Les transactions et transferts financiers
* La suppression ou la modification de données de production
* L'envoi de communications à des parties externes
* Toute opération ayant un impact significatif sur l'activité

In [103]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def send_email(email_address: str) -> str:
    """Simulate email handling for the HITL lesson; this never sends email."""
    return f"Simulation only: approval completed for {email_address}; no email was sent."

agent = create_agent(
    model=gemini_model,
    tools=[send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=InMemorySaver(),
    system_prompt="For an explicit request to simulate an email, call the send_email demo tool.",
)
config = {"configurable": {"thread_id": "tp5-hitl-demo"}}

# The tool is a local simulation and will not send a message.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Simulate sending an email to alex@example.com."}]},
    config=config,
)


In [104]:
pending_interrupts = result.get("__interrupt__", [])
print(pending_interrupts or "No simulated tool approval was requested.")


[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'email_address': 'alex@example.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'email_address': 'alex@example.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='4ee2bda2250bb2de2debb81ef3c80346', response_schema=None)]


In [105]:
if pending_interrupts:
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
else:
    print("Nothing to approve; the example remains a local simulation.")


In [106]:
if result.get("messages"):
    print(response_text(result["messages"][-1]))


The email simulation to alex@example.com has been executed.


# Garde-fous personnalisés

Pour des garde-fous plus sophistiqués, vous pouvez créer un middleware personnalisé qui s'exécute avant ou après l'exécution de l'agent. Cela vous donne un contrôle total sur la logique de validation, le filtrage du contenu et les vérifications de sécurité.

## Garde-fous avant l'agent

Utilisez les hooks « avant l'agent » pour valider les requêtes une seule fois, au début de chaque invocation. Cela est utile pour des vérifications au niveau de la session, comme l'authentification, la limitation du débit (rate limiting) ou le blocage des requêtes inappropriées avant le début de tout traitement.

In [107]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState
from langgraph.runtime import Runtime

banned_keywords = ["hack", "exploit", "malware"]

@tool
def search(query: str) -> str:
    """Use DuckDuckGo search if this optional tool is reached."""
    return web_search.invoke({"query": query, "num_results": 5})

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Block the demonstration request locally before the model is called."""
    if not state["messages"]:
        return None
    first_message = state["messages"][0]
    if first_message.type != "human":
        return None
    content = response_text(first_message).lower()
    if any(keyword in content for keyword in banned_keywords):
        return {
            "messages": [{"role": "assistant", "content": "Blocked locally by the content filter."}],
            "jump_to": "end",
        }
    return None

from langchain.agents import create_agent
agent = create_agent(model=gemini_model, tools=[search], middleware=[content_filter])

# This phrase is blocked by the local guardrail before Gemini or web search is called.
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})


In [108]:
print(response_text(result['messages'][-1]))

Blocked locally by the content filter.


## Garde-fous après l'agent

Utilisez les hooks « après l'agent » pour valider les sorties finales une seule fois avant de les renvoyer à l'utilisateur. 

Cela est utile pour les vérifications de sécurité fondées sur un modèle, la validation de la qualité ou les contrôles finaux de conformité sur la réponse complète de l'agent.

In [109]:
from langchain.agents.middleware import after_agent, AgentState
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from typing import Any

safety_model = gemini_model

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Ask Gemini to classify the final response as SAFE or UNSAFE."""
    if not state["messages"]:
        return None
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    safety_prompt = (
        "Classify this response as SAFE or UNSAFE. Reply with exactly one label.\n"
        f"Response: {response_text(last_message)}"
    )
    safety_result = safety_model.invoke([{"role": "user", "content": safety_prompt}])
    if "UNSAFE" in response_text(safety_result).upper():
        return {"messages": [AIMessage(content="I cannot provide that response. Please rephrase your request.")]}
    return None

from langchain.agents import create_agent
agent = create_agent(model=gemini_model, tools=[search], middleware=[safety_guardrail])

# Safe test input; no harmful how-to request is sent to the model.
result = agent.invoke({
    "messages": [{"role": "user", "content": "What are basic online safety practices?"}]
})


In [110]:
print(response_text(result['messages'][-1]))

Basic online safety practices are essential habits that help protect your personal information, financial data, and devices from cyber threats like hackers, scammers, and malware. 

Here are the foundational practices everyone should follow:

### 1. Master Your Passwords
*   **Use Strong, Unique Passwords:** Avoid simple passwords like "123456," "password," or your birthdate. Make them long (at least 12–16 characters) and mix uppercase letters, lowercase letters, numbers, and symbols.
*   **Never Reuse Passwords:** If a hacker steals your password from one website, they will try it on your email, bank, and social media accounts. 
*   **Use a Password Manager:** Instead of writing passwords down or trying to memorize dozens of them, use a reputable password manager (like Bitwarden, 1Password, or the one built into your browser/phone) to generate and store them securely.

### 2. Enable Multi-Factor Authentication (MFA / 2FA)
*   Turn on 2FA wherever it’s available (especially for email, 

In [111]:
from langchain.tools import tool
from langchain.messages import HumanMessage

@tool
def search_tool(query: str) -> str:
    """Expose the no-key DuckDuckGo search function as a Gemini tool."""
    return web_search.invoke({"query": query, "num_results": 5})


In [112]:
# Use the same Gemini client; no NVIDIA or OpenAI key is required.
llm = gemini_model


In [113]:
llm_with_tools = llm.bind_tools([search_tool])
resp = llm_with_tools.invoke([
    HumanMessage(content="What is the weather forecast for Marrakech today?")
])


In [114]:
print(resp.tool_calls if resp.tool_calls else response_text(resp))


[{'name': 'search_tool', 'args': {'query': 'Marrakech weather forecast today'}, 'id': 'call_96822', 'type': 'tool_call'}]


In [115]:
print("test")

test


# LangGraph